
# GVH Diagonal Cubic 0.3.1 — Real Observation Data Framework
## Provenance, intégrité, unités et interface commune des données réelles

**Auteur : Charlemagne O Laurince**

## Objectif

La Partie C commence par la transition :

\[
\boxed{\text{prototypes synthétiques}\rightarrow\text{données observationnelles documentées}}
\]

Ce notebook construit l’infrastructure commune pour enregistrer la provenance, la version, la licence, les unités, les checksums, les schémas et le statut réel de chaque jeu de données. Aucune donnée absente n’est inventée : elle reste marquée `NOT_LOADED`.


In [1]:

import json, hashlib, platform, sys
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
print("Python:", sys.version)
print("Plateforme:", platform.platform())


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Plateforme: Linux-6.6.122+-x86_64-with-glibc2.35



# 1. Arborescence

```text
gvh_diagonal_cubic/
├── data/
│   ├── raw/
│   ├── prepared/
│   ├── metadata/
│   └── checksums/
└── exports/
```

Les fichiers bruts sont immuables. Toute transformation produit un nouveau fichier dans `prepared/`.


In [2]:

PROJECT_ROOT=Path('.')
DATA_ROOT=PROJECT_ROOT/'data'
RAW_DIR=DATA_ROOT/'raw'; PREPARED_DIR=DATA_ROOT/'prepared'
METADATA_DIR=DATA_ROOT/'metadata'; CHECKSUM_DIR=DATA_ROOT/'checksums'
EXPORT_DIR=PROJECT_ROOT/'exports'
for d in [RAW_DIR,PREPARED_DIR,METADATA_DIR,CHECKSUM_DIR,EXPORT_DIR]: d.mkdir(parents=True,exist_ok=True)


# 2. Statuts et registre

In [3]:

ALLOWED_STATUSES={"PLANNED","NOT_LOADED","DOWNLOADED","INTEGRITY_VERIFIED","SCHEMA_VALIDATED","READY_FOR_ANALYSIS","REJECTED"}
@dataclass
class DatasetRecord:
    dataset_id:str; domain:str; title:str; source_organization:str; official_reference:str
    version:str="UNSPECIFIED"; access_date_utc:str="NOT_ACCESSED"; license_name:str="UNVERIFIED"
    citation_text:str="TO_BE_COMPLETED"; expected_format:str="UNKNOWN"
    raw_relative_path:str=""; prepared_relative_path:str=""
    expected_columns:list=field(default_factory=list); expected_units:dict=field(default_factory=dict)
    covariance_required:bool=False; checksum_sha256:str=""; status:str="NOT_LOADED"; notes:str=""
    def validate(self):
        if self.status not in ALLOWED_STATUSES: raise ValueError(self.status)
        if not self.dataset_id: raise ValueError('dataset_id vide')
        return True


In [4]:

DATASET_REGISTRY=[
DatasetRecord('EOS_NUCLEAR_TABLES','neutron_star_eos','Verified nuclear EOS tables','MULTIPLE — TO VERIFY','TO_BE_VERIFIED',expected_format='CSV/TXT/HDF5',raw_relative_path='data/raw/eos/',prepared_relative_path='data/prepared/eos/',expected_columns=['energy_density','pressure','baryon_density'],expected_units={'energy_density':'MUST_BE_DOCUMENTED','pressure':'MUST_BE_DOCUMENTED','baryon_density':'MUST_BE_DOCUMENTED'},notes='Une provenance et une licence par EOS.'),
DatasetRecord('NICER_MASS_RADIUS','neutron_star_structure','NICER mass-radius posterior products','NICER collaboration / official archive','TO_BE_VERIFIED',expected_format='HDF5/FITS/CSV',raw_relative_path='data/raw/nicer/',prepared_relative_path='data/prepared/nicer/',expected_columns=['mass_solar','radius_km','posterior_weight'],expected_units={'mass_solar':'solar_mass','radius_km':'km','posterior_weight':'dimensionless'},covariance_required=True),
DatasetRecord('MASSIVE_PULSARS','neutron_star_structure','High-mass pulsar measurements','Published timing collaborations','TO_BE_VERIFIED',expected_format='CSV',raw_relative_path='data/raw/pulsars/',prepared_relative_path='data/prepared/pulsars/',expected_columns=['source_name','mass_solar','mass_sigma_solar'],expected_units={'mass_solar':'solar_mass','mass_sigma_solar':'solar_mass'}),
DatasetRecord('GW_PUBLIC_STRAIN','gravitational_waves','Public calibrated GW strain','Official open GW archive','TO_BE_VERIFIED',expected_format='HDF5/GWF',raw_relative_path='data/raw/gw/strain/',prepared_relative_path='data/prepared/gw/strain/',expected_columns=['time','strain'],expected_units={'time':'s','strain':'dimensionless'}),
DatasetRecord('GW_EVENT_POSTERIORS','gravitational_waves','Public compact-binary posterior samples','Official collaboration data release','TO_BE_VERIFIED',expected_format='HDF5/JSON/CSV',raw_relative_path='data/raw/gw/posteriors/',prepared_relative_path='data/prepared/gw/posteriors/',expected_columns=['mass_1_source','mass_2_source','lambda_1','lambda_2','log_likelihood'],expected_units={'mass_1_source':'solar_mass','mass_2_source':'solar_mass','lambda_1':'dimensionless','lambda_2':'dimensionless','log_likelihood':'dimensionless'}),
DatasetRecord('PPN_SOLAR_SYSTEM','weak_field','Documented Solar-System and PPN constraints','Official mission/publication sources','TO_BE_VERIFIED',expected_format='CSV',raw_relative_path='data/raw/ppn/',prepared_relative_path='data/prepared/ppn/',expected_columns=['observable','value','sigma','reference'],expected_units={'observable':'label','value':'observable_specific','sigma':'same_as_value','reference':'citation'},covariance_required=True),
DatasetRecord('PANTHEON_PLUS','cosmology','Pantheon+ supernova compilation','Official Pantheon+ release','TO_BE_VERIFIED',expected_format='TXT/CSV',raw_relative_path='data/raw/cosmology/pantheon_plus/',prepared_relative_path='data/prepared/cosmology/pantheon_plus/',expected_columns=['redshift','distance_modulus'],expected_units={'redshift':'dimensionless','distance_modulus':'mag'},covariance_required=True),
DatasetRecord('BAO_COMPILATION','cosmology','Documented BAO measurements','Survey collaborations','TO_BE_VERIFIED',expected_format='CSV/TXT',raw_relative_path='data/raw/cosmology/bao/',prepared_relative_path='data/prepared/cosmology/bao/',expected_columns=['redshift','observable','value','sigma'],expected_units={'redshift':'dimensionless','observable':'label','value':'observable_specific','sigma':'same_as_value'},covariance_required=True),
DatasetRecord('HUBBLE_HZ','cosmology','Documented H(z) measurements','Published observational sources','TO_BE_VERIFIED',expected_format='CSV',raw_relative_path='data/raw/cosmology/hz/',prepared_relative_path='data/prepared/cosmology/hz/',expected_columns=['redshift','H','sigma_H','method','reference'],expected_units={'redshift':'dimensionless','H':'km/s/Mpc','sigma_H':'km/s/Mpc','method':'label','reference':'citation'}),
DatasetRecord('CMB_PRODUCTS','cosmology','Official CMB likelihood or map products','Official CMB mission archive','TO_BE_VERIFIED',expected_format='FITS/HDF5',raw_relative_path='data/raw/cosmology/cmb/',prepared_relative_path='data/prepared/cosmology/cmb/',covariance_required=True)
]
for r in DATASET_REGISTRY: r.validate()
registry_df=pd.DataFrame([asdict(r) for r in DATASET_REGISTRY])
registry_df[['dataset_id','domain','status']]


,dataset_id,domain,status
0,EOS_NUCLEAR_TABLES,neutron_star_eos,NOT_LOADED
1,NICER_MASS_RADIUS,neutron_star_structure,NOT_LOADED
2,MASSIVE_PULSARS,neutron_star_structure,NOT_LOADED
3,GW_PUBLIC_STRAIN,gravitational_waves,NOT_LOADED
4,GW_EVENT_POSTERIORS,gravitational_waves,NOT_LOADED
5,PPN_SOLAR_SYSTEM,weak_field,NOT_LOADED
6,PANTHEON_PLUS,cosmology,NOT_LOADED
7,BAO_COMPILATION,cosmology,NOT_LOADED
8,HUBBLE_HZ,cosmology,NOT_LOADED
9,CMB_PRODUCTS,cosmology,NOT_LOADED


# 3. Intégrité SHA-256 et présence locale

In [5]:

def sha256_file(path,chunk_size=1024*1024):
    path=Path(path); h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(chunk_size),b''): h.update(chunk)
    return h.hexdigest()

def find_local_files(record,project_root=PROJECT_ROOT):
    p=project_root/record.raw_relative_path
    if not p.exists(): return []
    return [p] if p.is_file() else sorted(x for x in p.rglob('*') if x.is_file())

presence_df=pd.DataFrame([{'dataset_id':r.dataset_id,'status':r.status,'raw_path':r.raw_relative_path,'file_count':len(find_local_files(r)),'locally_present':bool(find_local_files(r))} for r in DATASET_REGISTRY])
presence_df


,dataset_id,status,raw_path,file_count,locally_present
0,EOS_NUCLEAR_TABLES,NOT_LOADED,data/raw/eos/,0,False
1,NICER_MASS_RADIUS,NOT_LOADED,data/raw/nicer/,0,False
2,MASSIVE_PULSARS,NOT_LOADED,data/raw/pulsars/,0,False
3,GW_PUBLIC_STRAIN,NOT_LOADED,data/raw/gw/strain/,0,False
4,GW_EVENT_POSTERIORS,NOT_LOADED,data/raw/gw/posteriors/,0,False
5,PPN_SOLAR_SYSTEM,NOT_LOADED,data/raw/ppn/,0,False
6,PANTHEON_PLUS,NOT_LOADED,data/raw/cosmology/pantheon_plus/,0,False
7,BAO_COMPILATION,NOT_LOADED,data/raw/cosmology/bao/,0,False
8,HUBBLE_HZ,NOT_LOADED,data/raw/cosmology/hz/,0,False
9,CMB_PRODUCTS,NOT_LOADED,data/raw/cosmology/cmb/,0,False


# 4. Validation de schéma tabulaire

In [6]:

def validate_tabular_schema(table,expected_columns):
    missing=sorted(set(expected_columns)-set(table.columns))
    numeric=table.select_dtypes(include=[np.number]).columns
    return {'row_count':len(table),'missing_columns':missing,'extra_columns':sorted(set(table.columns)-set(expected_columns)),'duplicate_rows':int(table.duplicated().sum()),'null_counts':table.isna().sum().astype(int).to_dict(),'nonfinite_counts':{c:int((~np.isfinite(table[c].to_numpy(float))).sum()) for c in numeric},'schema_pass':len(missing)==0 and len(table)>0}


# 5. Registre des unités et transformations

In [7]:

M_SUN_SI=1.98847e30; MPC_SI=3.085677581491367e22
UNIT_CONVERSION_REGISTRY=pd.DataFrame([
{'source_unit':'solar_mass','target_unit':'kg','conversion':'x*M_SUN_SI','status':'DEFINED'},
{'source_unit':'km','target_unit':'m','conversion':'x*1000','status':'DEFINED'},
{'source_unit':'km/s/Mpc','target_unit':'1/s','conversion':'x*1000/MPC_SI','status':'DEFINED'},
{'source_unit':'MeV/fm^3','target_unit':'Pa','conversion':'REQUIRES_VERIFIED_CONSTANTS','status':'NOT_ACTIVATED'}])
def solar_mass_to_kg(x): return np.asarray(x,float)*M_SUN_SI
def km_to_m(x): return np.asarray(x,float)*1000.0
def hubble_km_s_mpc_to_inverse_seconds(x): return np.asarray(x,float)*1000.0/MPC_SI
UNIT_CONVERSION_REGISTRY


,source_unit,target_unit,conversion,status
0,solar_mass,kg,x*M_SUN_SI,DEFINED
1,km,m,x*1000,DEFINED
2,km/s/Mpc,1/s,x*1000/MPC_SI,DEFINED
3,MeV/fm^3,Pa,REQUIRES_VERIFIED_CONSTANTS,NOT_ACTIVATED


In [8]:

TRANSFORMATION_LOG_COLUMNS=['timestamp_utc','dataset_id','input_path','output_path','operation','parameters_json','input_sha256','output_sha256','software_version']
transformation_log_df=pd.DataFrame(columns=TRANSFORMATION_LOG_COLUMNS)
def append_transformation_log(log_df,dataset_id,input_path,output_path,operation,parameters,software_version='GVH_0.3.1'):
    ip,op=Path(input_path),Path(output_path)
    row={'timestamp_utc':datetime.now(timezone.utc).isoformat(),'dataset_id':dataset_id,'input_path':str(ip),'output_path':str(op),'operation':operation,'parameters_json':json.dumps(parameters,sort_keys=True),'input_sha256':sha256_file(ip) if ip.exists() and ip.is_file() else 'NOT_AVAILABLE','output_sha256':sha256_file(op) if op.exists() and op.is_file() else 'NOT_AVAILABLE','software_version':software_version}
    return pd.concat([log_df,pd.DataFrame([row])],ignore_index=True)


# 6. Métadonnées obligatoires et readiness gate

In [9]:

REQUIRED_METADATA_FIELDS=['dataset_id','title','source_organization','official_reference','version','access_date_utc','license_name','citation_text','expected_format','status']
PLACEHOLDERS={'','UNSPECIFIED','UNVERIFIED','TO_BE_COMPLETED','TO_BE_VERIFIED','NOT_ACCESSED'}
def validate_record_metadata(record):
    d=asdict(record); miss=[f for f in REQUIRED_METADATA_FIELDS if d.get(f) is None or str(d.get(f)) in PLACEHOLDERS]
    return {'dataset_id':record.dataset_id,'metadata_ready':not miss,'missing_or_placeholder':miss}
def readiness_gate(file_present,checksum_recorded,schema_validated,units_documented,license_verified):
    c=locals().copy(); c['ready_for_analysis']=all(c.values()); return c
metadata_audit_df=pd.DataFrame([validate_record_metadata(r) for r in DATASET_REGISTRY])
metadata_audit_df


,dataset_id,metadata_ready,missing_or_placeholder
0,EOS_NUCLEAR_TABLES,False,"[official_reference, version, access_date_utc,..."
1,NICER_MASS_RADIUS,False,"[official_reference, version, access_date_utc,..."
2,MASSIVE_PULSARS,False,"[official_reference, version, access_date_utc,..."
3,GW_PUBLIC_STRAIN,False,"[official_reference, version, access_date_utc,..."
4,GW_EVENT_POSTERIORS,False,"[official_reference, version, access_date_utc,..."
5,PPN_SOLAR_SYSTEM,False,"[official_reference, version, access_date_utc,..."
6,PANTHEON_PLUS,False,"[official_reference, version, access_date_utc,..."
7,BAO_COMPILATION,False,"[official_reference, version, access_date_utc,..."
8,HUBBLE_HZ,False,"[official_reference, version, access_date_utc,..."
9,CMB_PRODUCTS,False,"[official_reference, version, access_date_utc,..."


# 7. Manifeste de fichiers

In [10]:

FILE_MANIFEST_COLUMNS=['dataset_id','relative_path','file_name','file_size_bytes','sha256','format','source_version','access_date_utc','license_name','status']
def build_file_manifest(record):
    rows=[]
    for p in find_local_files(record):
        rows.append({'dataset_id':record.dataset_id,'relative_path':str(p.relative_to(PROJECT_ROOT)),'file_name':p.name,'file_size_bytes':p.stat().st_size,'sha256':sha256_file(p),'format':p.suffix.lower(),'source_version':record.version,'access_date_utc':record.access_date_utc,'license_name':record.license_name,'status':record.status})
    return pd.DataFrame(rows,columns=FILE_MANIFEST_COLUMNS)
frames=[build_file_manifest(r) for r in DATASET_REGISTRY]
file_manifest_df=pd.concat([f for f in frames if not f.empty],ignore_index=True) if any(not f.empty for f in frames) else pd.DataFrame(columns=FILE_MANIFEST_COLUMNS)
file_manifest_df


,dataset_id,relative_path,file_name,file_size_bytes,sha256,format,source_version,access_date_utc,license_name,status


# 8. Interface commune préparée

In [11]:

COMMON_PREPARED_COLUMNS=['record_id','observable','value','sigma','unit','reference','quality_flag']
def validate_common_prepared_table(table):
    base=validate_tabular_schema(table,COMMON_PREPARED_COLUMNS)
    sigma=pd.to_numeric(table['sigma'],errors='coerce') if 'sigma' in table else pd.Series(dtype=float)
    base['positive_sigma']=bool((sigma.dropna()>0).all()) if len(sigma) else False
    base['unique_record_id']=bool(table['record_id'].is_unique) if 'record_id' in table else False
    base['prepared_table_pass']=base['schema_pass'] and base['positive_sigma'] and base['unique_record_id']
    return base
synthetic_template_df=pd.DataFrame([
{'record_id':'EXAMPLE-001','observable':'example_quantity','value':1.0,'sigma':0.1,'unit':'arbitrary','reference':'SYNTHETIC_TEMPLATE','quality_flag':'TEST_ONLY'},
{'record_id':'EXAMPLE-002','observable':'example_quantity','value':1.2,'sigma':0.1,'unit':'arbitrary','reference':'SYNTHETIC_TEMPLATE','quality_flag':'TEST_ONLY'}])
synthetic_template_audit=validate_common_prepared_table(synthetic_template_df)
synthetic_template_audit


{'row_count': 2,
 'missing_columns': [],
 'extra_columns': [],
 'duplicate_rows': 0,
 'null_counts': {'record_id': 0,
  'observable': 0,
  'value': 0,
  'sigma': 0,
  'unit': 0,
  'reference': 0,
  'quality_flag': 0},
 'nonfinite_counts': {'value': 0, 'sigma': 0},
 'schema_pass': True,
 'positive_sigma': True,
 'unique_record_id': True,
 'prepared_table_pass': True}

# 9. Disponibilité et portes par domaine

In [12]:

availability=[]
for r in DATASET_REGISTRY:
    files=find_local_files(r); meta=validate_record_metadata(r)
    units=bool(r.expected_units) and all(v not in {'','MUST_BE_DOCUMENTED','observable_specific'} for v in r.expected_units.values())
    ready=readiness_gate(bool(files),bool(r.checksum_sha256),r.status in {'SCHEMA_VALIDATED','READY_FOR_ANALYSIS'},units,r.license_name not in {'','UNVERIFIED'})
    availability.append({'dataset_id':r.dataset_id,'domain':r.domain,'status':r.status,'file_count':len(files),'metadata_ready':meta['metadata_ready'],'units_documented':units,'license_verified':ready['license_verified'],'ready_for_analysis':ready['ready_for_analysis']})
availability_df=pd.DataFrame(availability)
availability_df


,dataset_id,domain,status,file_count,metadata_ready,units_documented,license_verified,ready_for_analysis
0,EOS_NUCLEAR_TABLES,neutron_star_eos,NOT_LOADED,0,False,False,False,False
1,NICER_MASS_RADIUS,neutron_star_structure,NOT_LOADED,0,False,True,False,False
2,MASSIVE_PULSARS,neutron_star_structure,NOT_LOADED,0,False,True,False,False
3,GW_PUBLIC_STRAIN,gravitational_waves,NOT_LOADED,0,False,True,False,False
4,GW_EVENT_POSTERIORS,gravitational_waves,NOT_LOADED,0,False,True,False,False
5,PPN_SOLAR_SYSTEM,weak_field,NOT_LOADED,0,False,False,False,False
6,PANTHEON_PLUS,cosmology,NOT_LOADED,0,False,True,False,False
7,BAO_COMPILATION,cosmology,NOT_LOADED,0,False,False,False,False
8,HUBBLE_HZ,cosmology,NOT_LOADED,0,False,True,False,False
9,CMB_PRODUCTS,cosmology,NOT_LOADED,0,False,False,False,False


In [13]:

DOMAIN_GATES={
'neutron_star_eos':['EOS_NUCLEAR_TABLES'],
'neutron_star_structure':['NICER_MASS_RADIUS','MASSIVE_PULSARS'],
'gravitational_waves':['GW_PUBLIC_STRAIN','GW_EVENT_POSTERIORS'],
'weak_field':['PPN_SOLAR_SYSTEM'],
'cosmology':['PANTHEON_PLUS','BAO_COMPILATION','HUBBLE_HZ','CMB_PRODUCTS']}
def evaluate_domain_gate(domain):
    req=DOMAIN_GATES[domain]; sub=availability_df[availability_df.dataset_id.isin(req)]
    return {'domain':domain,'required_dataset_ids':req,'ready_dataset_ids':sub.loc[sub.ready_for_analysis,'dataset_id'].tolist(),'gate_pass':len(sub)==len(req) and bool(sub.ready_for_analysis.all())}
domain_gate_df=pd.DataFrame([evaluate_domain_gate(d) for d in DOMAIN_GATES])
domain_gate_df


,domain,required_dataset_ids,ready_dataset_ids,gate_pass
0,neutron_star_eos,[EOS_NUCLEAR_TABLES],[],False
1,neutron_star_structure,"[NICER_MASS_RADIUS, MASSIVE_PULSARS]",[],False
2,gravitational_waves,"[GW_PUBLIC_STRAIN, GW_EVENT_POSTERIORS]",[],False
3,weak_field,[PPN_SOLAR_SYSTEM],[],False
4,cosmology,"[PANTHEON_PLUS, BAO_COMPILATION, HUBBLE_HZ, CM...",[],False


# 10. Validation automatique

In [14]:

validation_df=pd.DataFrame([
{'test':'nonempty registry','status':'PASS' if len(DATASET_REGISTRY)>0 else 'FAIL'},
{'test':'unique dataset ids','status':'PASS' if registry_df.dataset_id.is_unique else 'FAIL'},
{'test':'valid statuses','status':'PASS' if registry_df.status.isin(ALLOWED_STATUSES).all() else 'FAIL'},
{'test':'raw/prepared separation','status':'PASS' if RAW_DIR.resolve()!=PREPARED_DIR.resolve() else 'FAIL'},
{'test':'synthetic template interface','status':'PASS' if synthetic_template_audit['prepared_table_pass'] else 'FAIL'},
{'test':'manifest schema','status':'PASS' if set(FILE_MANIFEST_COLUMNS).issubset(file_manifest_df.columns) else 'FAIL'},
{'test':'transformation log schema','status':'PASS' if set(TRANSFORMATION_LOG_COLUMNS).issubset(transformation_log_df.columns) else 'FAIL'},
{'test':'no unsupported real-data claim','status':'PASS' if not availability_df.ready_for_analysis.any() else 'CHECK'}])
overall_status='PASS-REAL-DATA-FRAMEWORK' if (validation_df.status=='PASS').all() else 'CHECK'
print('STATUT:',overall_status)
print('JEUX ENREGISTRÉS:',len(DATASET_REGISTRY))
print('JEUX PRÊTS:',int(availability_df.ready_for_analysis.sum()))
validation_df


STATUT: PASS-REAL-DATA-FRAMEWORK
JEUX ENREGISTRÉS: 10
JEUX PRÊTS: 0


,test,status
0,nonempty registry,PASS
1,unique dataset ids,PASS
2,valid statuses,PASS
3,raw/prepared separation,PASS
4,synthetic template interface,PASS
5,manifest schema,PASS
6,transformation log schema,PASS
7,no unsupported real-data claim,PASS


# 11. Feuille de route de la Partie C

In [15]:

PART_C_ROADMAP=pd.DataFrame([
{'step':'0.3.2','notebook':'GVH_Diagonal_Cubic_0.3.2_Verified_Nuclear_EOS_Ingestion.ipynb','domain':'neutron_star_eos','priority':'P0'},
{'step':'0.3.3','notebook':'GVH_Diagonal_Cubic_0.3.3_Realistic_Mass_Radius_Constraints.ipynb','domain':'neutron_star_structure','priority':'P0'},
{'step':'0.3.4','notebook':'GVH_Diagonal_Cubic_0.3.4_Solar_System_PPN_Real_Constraints.ipynb','domain':'weak_field','priority':'P0'},
{'step':'0.3.5','notebook':'GVH_Diagonal_Cubic_0.3.5_Public_GW_Event_Data_Pipeline.ipynb','domain':'gravitational_waves','priority':'P1'},
{'step':'0.3.6','notebook':'GVH_Diagonal_Cubic_0.3.6_Real_GW_Likelihood_GR_GVH.ipynb','domain':'gravitational_waves','priority':'P1'},
{'step':'0.3.7','notebook':'GVH_Diagonal_Cubic_0.3.7_Cosmology_Real_Data_Interface.ipynb','domain':'cosmology','priority':'P1'},
{'step':'0.3.8','notebook':'GVH_Diagonal_Cubic_0.3.8_Real_Cross_Domain_Joint_Constraints.ipynb','domain':'joint_constraints','priority':'P1'}])
PART_C_ROADMAP


,step,notebook,domain,priority
0,0.3.2,GVH_Diagonal_Cubic_0.3.2_Verified_Nuclear_EOS_...,neutron_star_eos,P0
1,0.3.3,GVH_Diagonal_Cubic_0.3.3_Realistic_Mass_Radius...,neutron_star_structure,P0
2,0.3.4,GVH_Diagonal_Cubic_0.3.4_Solar_System_PPN_Real...,weak_field,P0
3,0.3.5,GVH_Diagonal_Cubic_0.3.5_Public_GW_Event_Data_...,gravitational_waves,P1
4,0.3.6,GVH_Diagonal_Cubic_0.3.6_Real_GW_Likelihood_GR...,gravitational_waves,P1
5,0.3.7,GVH_Diagonal_Cubic_0.3.7_Cosmology_Real_Data_I...,cosmology,P1
6,0.3.8,GVH_Diagonal_Cubic_0.3.8_Real_Cross_Domain_Joi...,joint_constraints,P1


# 12. Export

In [16]:

def jsonable_registry(df):
    out=df.copy()
    for c in ['expected_columns','expected_units']:
        out[c]=out[c].apply(lambda v:json.dumps(v,ensure_ascii=False,sort_keys=True))
    return out
jsonable_registry(registry_df).to_csv(EXPORT_DIR/'GVH_Diagonal_Cubic_0.3.1_Dataset_Registry.csv',index=False)
availability_df.to_csv(EXPORT_DIR/'GVH_Diagonal_Cubic_0.3.1_Availability.csv',index=False)
domain_gate_df.to_csv(EXPORT_DIR/'GVH_Diagonal_Cubic_0.3.1_Domain_Gates.csv',index=False)
metadata_audit_df.to_csv(EXPORT_DIR/'GVH_Diagonal_Cubic_0.3.1_Metadata_Audit.csv',index=False)
file_manifest_df.to_csv(EXPORT_DIR/'GVH_Diagonal_Cubic_0.3.1_File_Manifest.csv',index=False)
UNIT_CONVERSION_REGISTRY.to_csv(EXPORT_DIR/'GVH_Diagonal_Cubic_0.3.1_Unit_Registry.csv',index=False)
transformation_log_df.to_csv(EXPORT_DIR/'GVH_Diagonal_Cubic_0.3.1_Transformation_Log.csv',index=False)
synthetic_template_df.to_csv(EXPORT_DIR/'GVH_Diagonal_Cubic_0.3.1_Prepared_Table_Template.csv',index=False)
validation_df.to_csv(EXPORT_DIR/'GVH_Diagonal_Cubic_0.3.1_Validation.csv',index=False)
PART_C_ROADMAP.to_csv(EXPORT_DIR/'GVH_Diagonal_Cubic_0.3.1_Part_C_Roadmap.csv',index=False)
metadata={'notebook':'GVH_Diagonal_Cubic_0.3.1_Real_Observation_Data_Framework','status':overall_status,'execution_time_utc':datetime.now(timezone.utc).isoformat(),'dataset_count':len(DATASET_REGISTRY),'ready_for_analysis_count':int(availability_df.ready_for_analysis.sum()),'real_data_analyzed':False,'next_notebook':'GVH_Diagonal_Cubic_0.3.2_Verified_Nuclear_EOS_Ingestion.ipynb'}
with (EXPORT_DIR/'GVH_Diagonal_Cubic_0.3.1_Metadata.json').open('w',encoding='utf-8') as f: json.dump(metadata,f,indent=2,ensure_ascii=False)
print('Exports terminés dans',EXPORT_DIR)


Exports terminés dans exports



# Conclusion

La Partie C dispose maintenant d’une porte d’entrée reproductible :

\[
\boxed{
\text{source officielle}
\rightarrow
\text{fichier brut}
\rightarrow
\text{checksum}
\rightarrow
\text{schéma}
\rightarrow
\text{unités}
\rightarrow
\text{donnée préparée}
}
\]

Le framework est construit, mais aucune donnée observationnelle réelle n’est encore analysée.

## Étape suivante

`GVH_Diagonal_Cubic_0.3.2_Verified_Nuclear_EOS_Ingestion.ipynb`
